# 01A Retrieve EUR-Lex Documents

This workbook reconstructs the main EUR-Lex retrieval layer around the original WebService method used in `NID_Retrieval_Pipeline_EURLEX.ipynb`.

The notebook is intentionally workbook-style: you can edit query terms and retrieval parameters directly here, inspect intermediate job tables and raw rows, inspect one real SOAP response, and then run the full retrieval with explicit fallback only at the nearest retrieval-output level.


# Setup

## Editable parameters in this notebook

Primary edit points:
- primary EUR-Lex search terms
- translated terms
- whether to include translated jobs and NIM scope
- fields searched in the WebService query (`TI`, `TE`)
- chunk size, page size, max pages
- whether to attempt live SOAP/WebService retrieval
- whether to run the parser debug probe and save raw XML
- whether to attempt live batch full-text retrieval
- full-text rate limits, retry counts, and cache usage


In [20]:
from __future__ import annotations

from pathlib import Path
import sys
import os
import importlib

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "docs_df_cleaned").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

PIPE_DIR = ROOT / "analysis_pipeline"
PIPE_OUT = PIPE_DIR / "outputs"
PIPE_OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 60)
SAVE_OUTPUTS = True


def preview_df(name: str, df: pd.DataFrame, n: int = 5) -> None:
    print(f"{name}: shape={df.shape}")
    display(df.head(n))


def write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if SAVE_OUTPUTS:
        df.to_csv(path, index=index)
    print(f"{'Wrote' if SAVE_OUTPUTS else 'Would write'}: {path}")
    print(f"Shape: {df.shape}")
    display(df.head())


import analysis_pipeline.functions.eurlex_retrieval as eurlex_retrieval_module
import analysis_pipeline.functions.retrieval_queries as retrieval_queries_module
importlib.reload(eurlex_retrieval_module)
importlib.reload(retrieval_queries_module)

from analysis_pipeline.functions.retrieval_queries import (
    SEARCH_TERMS_PRIMARY,
    TRANSLATED_TERMS_PRIMARY,
)
from analysis_pipeline.functions.eurlex_retrieval import (
    build_eurlex_jobs,
    build_job_chunks,
    build_expert_query,
    build_soap_envelope,
    post_eurlex_ws,
    parse_searchresults,
    build_result_diagnostics,
    extract_result_snippets,
    run_eurlex_jobs,
    load_cached_eu_rows,
    reconstruct_eu_rows_from_corpus,
    build_eu_doc_tables,
    lang_candidates_from_row,
    celex_variants,
    cellar_celex_url,
    annotate_celex_types,
    filter_celex_types_for_fulltext,
    batch_fetch_eurlex_fulltext,
    summarize_fulltext_cache_state,
    load_failed_fulltext_attempts,
    fulltext_cache_validation_diagnostics,
    load_eu_fulltext_docs,
    trace_eurlex_fulltext_decision,
)

CANONICAL_ALL_DOCS = ROOT / "docs_df_cleaned" / "all_docs_df.csv"
CACHED_ROWS = PIPE_OUT / "01A_all_eu_rows.csv"
OUT_FULLTEXT = PIPE_OUT / "01A_eu_all_fulltext_docs_new.csv"
DEBUG_XML_PATH = PIPE_OUT / "page1_raw_response.xml"
FULLTEXT_CACHE_DIR = PIPE_OUT / "eurlex_all_fulltext_cache"
FULLTEXT_INDEX_OUT = PIPE_OUT / "eurlex_all_fulltext_index.csv"
# CACHED_ROWS = PIPE_OUT / "01A_all_eu_en_rows.csv"
# OUT_FULLTEXT = PIPE_OUT / "01A_eu_en_fulltext_docs.csv"
# DEBUG_XML_PATH = PIPE_OUT / "page1_raw_response.xml"
# FULLTEXT_CACHE_DIR = PIPE_OUT / "eurlex_en_fulltext_cache"
# FULLTEXT_INDEX_OUT = PIPE_OUT / "eurlex_en_fulltext_index.csv"

# OUT_ROWS = PIPE_OUT / "01A_all_eu_rows.csv"
# OUT_RAW = PIPE_OUT / "01A_eu_all_df_raw.csv"
# OUT_DOCS = PIPE_OUT / "01A_eu_all_df_docs.csv"

os.environ["EURLEX_USER"] = "n00esrcw"
os.environ["EURLEX_WEB_PASS"] = "FPfgrb83td0"


## Query terms used in this notebook

These cells surface the original primary EUR-Lex retrieval vocabulary directly in the notebook.

Note: the original primary WebService term list did **not** include `net-gain` / `net gain`. Those belonged to the later full-text/body matching layer and remain separate.


In [8]:
SEARCH_TERMS = list(SEARCH_TERMS_PRIMARY)
TRANSLATED_TERMS = {lang: list(terms) for lang, terms in TRANSLATED_TERMS_PRIMARY.items()}

INCLUDE_TRANSLATIONS = True
INCLUDE_NIM_SCOPE = True
FIELDS = ("TI", "TE")
TERMS_PER_QUERY = 12
PAGE_SIZE = 100
PAGE_SIZE_CANDIDATES = (100, 50, 25)
MAX_PAGES = 100
MIN_INTERVAL_S = 1.6
TIMEOUT_S = 45
RETRY_5XX = 3
RUN_LIVE = True
ALLOW_FALLBACK = False
RUN_PARSER_DEBUG_PROBE = True
DEBUG_PAGE_SIZE = 10
DEBUG_TERM_LIMIT = 3
RUN_LIVE_FULLTEXT = True
FULLTEXT_MAX_DOCS = None
FULLTEXT_CELEX_FILTER_MODE = "all"  # all | supported_only | sector_3_only | sector_0_and_3
FULLTEXT_EXCLUDE_DESCRIPTORS = []
FULLTEXT_MIN_INTERVAL_S = 2.0
FULLTEXT_TIMEOUT_S = 45
FULLTEXT_RETRIES = 4
FULLTEXT_USE_CACHE = False
FULLTEXT_VERBOSE = True
FULLTEXT_TRACE_ROUTES = False
FULLTEXT_RESUME = True
FULLTEXT_RETRY_FAILURES = True
FULLTEXT_PROGRESS_EVERY = 5
FULLTEXT_CACHE_EVERY = 50
FULLTEXT_SUCCESS_MIN_CHARS = 500
FALLBACK_FILL_MISSING_TEXT = False
DEBUG = True

os.environ["EURLEX_USER"] = "n00esrcw"
os.environ["EURLEX_WEB_PASS"] = "FPfgrb83td0"

print("Primary search terms")
display(pd.DataFrame({"term": SEARCH_TERMS}))
print("Translated terms preview")
display(
    pd.DataFrame(
        [(lang, term) for lang, terms in TRANSLATED_TERMS.items() for term in terms],
        columns=["language", "term"],
    ).head(25)
)

FULLTEXT_TRACE_SINGLE_TIMEOUT_S = 25

FULLTEXT_TRACE_CELEX_LIST = ["01996L0061", "02002D0272", "02003L0087"]
FULLTEXT_TRACE_SINGLE_TIMEOUT_S = 25


Primary search terms


,term
0,nature-positive
1,nature positive
2,nature inclusive
3,nature-inclusive
4,nature-inclusive
5,nature inclusive design
6,nature based solutions
7,nature-based solutions
8,biodiversity net gain
9,biodiversity net-gain


Translated terms preview


,language,term
0,bg,интегриран с природата дизайн
1,bg,природосъобразен дизайн
2,bg,природопозитивен
3,bg,природосъобразни решения
4,bg,стратегия за биологичното разнообразие
5,bg,стратегия на ЕС за биологичното разнообразие
6,bg,възстановяване на природата
7,bg,възстановяване на екосистемите
8,cs,návrh zahrnující přírodu
9,cs,pozitivní pro přírodu


# Query EU docs based on search terms

## Build EUR-Lex jobs


In [3]:
jobs = build_eurlex_jobs(include_translations=INCLUDE_TRANSLATIONS, include_nim=INCLUDE_NIM_SCOPE)

if not INCLUDE_TRANSLATIONS:
    for job in jobs:
        if job["lang"] != "en":
            job["terms"] = []

for job in jobs:
    if job["lang"] == "en":
        job["terms"] = list(SEARCH_TERMS)
    elif INCLUDE_TRANSLATIONS:
        job["terms"] = list(TRANSLATED_TERMS.get(job["lang"], []))

query_jobs = pd.DataFrame(jobs)
query_jobs["term_count"] = query_jobs["terms"].map(len)
preview_df("query_jobs", query_jobs, n=10)
display(query_jobs[["scope", "lang", "expert_scope", "term_count"]].head(20))


query_jobs: shape=(50, 5)


,scope,expert_scope,lang,terms,term_count
0,ALL_ALL,DTS_SUBDOM = ALL_ALL,en,"[nature-positive, nature positive, nature inclusive, nature-inclusive, nature-inclusive, nature inclusive design, nature based solutions, nature-based solutions, biodiversity net gain, biodiversity net-gain, biodiver...",14
1,ALL_ALL,DTS_SUBDOM = ALL_ALL,bg,"[интегриран с природата дизайн, природосъобразен дизайн, природопозитивен, природосъобразни решения, стратегия за биологичното разнообразие, стратегия на ЕС за биологичното разнообразие, възстановяване на природата, ...",8
2,ALL_ALL,DTS_SUBDOM = ALL_ALL,cs,"[návrh zahrnující přírodu, pozitivní pro přírodu, řešení založená na přírodě, strategie v oblasti biologické rozmanitosti, strategie EU v oblasti biologické rozmanitosti, obnova přírody, obnova ekosystémů]",7
3,ALL_ALL,DTS_SUBDOM = ALL_ALL,da,"[naturinkluderende, naturinkluderende design, naturpositiv, naturbaserede løsninger, biodiversitetsstrategi, EU's biodiversitetsstrategi, genopretning af natur, genopretning af biodiversitet]",8
4,ALL_ALL,DTS_SUBDOM = ALL_ALL,de,"[naturverträgliche Planung, naturpositiv, naturbasierte Lösungen, naturbasierte Lösung, Netto-Gewinn, Netto-Gewinn für die Biodiversität, Biodiversitätsstrategie, EU-Biodiversitätsstrategie, Wiederherstellung der Nat...",10
5,ALL_ALL,DTS_SUBDOM = ALL_ALL,el,"[φιλικός προς τη φύση σχεδιασμός, θετικός για τη φύση, λύσεις βασισμένες στη φύση, καθαρό περιβαλλοντικό κέρδος, καθαρό κέρδος για τη βιοποικιλότητα, στρατηγική για τη βιοποικιλότητα, στρατηγική της ΕΕ για τη βιοποικ...",9
6,ALL_ALL,DTS_SUBDOM = ALL_ALL,es,"[diseño respetuoso con la naturaleza, positivo para la naturaleza, soluciones basadas en la naturaleza, ganancia neta, ganancia neta en biodiversidad, ganancia neta para la biodiversidad, estrategia de biodiversidad,...",11
7,ALL_ALL,DTS_SUBDOM = ALL_ALL,et,"[loodusega arvestav disain, looduspositiivne, looduspõhised lahendused, looduspõhine lahendus, elurikkuse strateegia, ELi elurikkuse strateegia, looduse taastamine, ökosüsteemide taastamine]",8
8,ALL_ALL,DTS_SUBDOM = ALL_ALL,fi,"[luontoystävällinen suunnittelu, luontopositiivinen, luontoon perustuvat ratkaisut, luontopohjaiset ratkaisut, biodiversiteettistrategia, EU:n biodiversiteettistrategia, luonnon ennallistaminen, ekosysteemien ennalli...",8
9,ALL_ALL,DTS_SUBDOM = ALL_ALL,fr,"[aménagement respectueux de la nature, positif pour la nature, solutions fondées sur la nature, solution fondée sur la nature, gain net, gain net pour la biodiversité, stratégie pour la biodiversité, stratégie de l'U...",10


,scope,lang,expert_scope,term_count
0,ALL_ALL,en,DTS_SUBDOM = ALL_ALL,14
1,ALL_ALL,bg,DTS_SUBDOM = ALL_ALL,8
2,ALL_ALL,cs,DTS_SUBDOM = ALL_ALL,7
3,ALL_ALL,da,DTS_SUBDOM = ALL_ALL,8
4,ALL_ALL,de,DTS_SUBDOM = ALL_ALL,10
5,ALL_ALL,el,DTS_SUBDOM = ALL_ALL,9
6,ALL_ALL,es,DTS_SUBDOM = ALL_ALL,11
7,ALL_ALL,et,DTS_SUBDOM = ALL_ALL,8
8,ALL_ALL,fi,DTS_SUBDOM = ALL_ALL,8
9,ALL_ALL,fr,DTS_SUBDOM = ALL_ALL,10


## Build query chunks


In [4]:
query_chunks = build_job_chunks(jobs, fields=FIELDS, terms_per_query=TERMS_PER_QUERY)
preview_df("query_chunks", query_chunks, n=14)
display(query_chunks[["job_index", "chunk_index", "scope", "lang", "term_count", "fields", "term_group"]].head(20))


query_chunks: shape=(54, 10)


,job_index,chunk_index,scope,lang,expert_scope,fields,term_count,terms,term_group,expert_query
0,1,1,ALL_ALL,en,DTS_SUBDOM = ALL_ALL,"TI, TE",12,"[nature-positive, nature positive, nature inclusive, nature-inclusive, nature-inclusive, nature inclusive design, nature based solutions, nature-based solutions, biodiversity net gain, biodiversity net-gain, biodiver...",nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,"DTS_SUBDOM = ALL_ALL AND (((TI ~ ""nature-positive"" OR TE ~ ""nature-positive"")) OR ((TI ~ ""nature positive"" OR TE ~ ""nature positive"")) OR ((TI ~ ""nature inclusive"" OR TE ~ ""nature inclusive"")) OR ((TI ~ ""nature-inclu..."
1,1,2,ALL_ALL,en,DTS_SUBDOM = ALL_ALL,"TI, TE",2,"[nature repair, biodiversity strategy]",nature repair | biodiversity strategy,"DTS_SUBDOM = ALL_ALL AND (((TI ~ ""nature repair"" OR TE ~ ""nature repair"")) OR ((TI ~ ""biodiversity strategy"" OR TE ~ ""biodiversity strategy"")))"
2,2,1,ALL_ALL,bg,DTS_SUBDOM = ALL_ALL,"TI, TE",8,"[интегриран с природата дизайн, природосъобразен дизайн, природопозитивен, природосъобразни решения, стратегия за биологичното разнообразие, стратегия на ЕС за биологичното разнообразие, възстановяване на природата, ...",интегриран с природата дизайн | природосъобразен дизайн | природопозитивен | природосъобразни решения | стратегия за биологичното разнообразие | стратегия на ЕС за биологичното разнообразие | възстановяване на природ...,"DTS_SUBDOM = ALL_ALL AND (((TI ~ ""интегриран с природата дизайн"" OR TE ~ ""интегриран с природата дизайн"")) OR ((TI ~ ""природосъобразен дизайн"" OR TE ~ ""природосъобразен дизайн"")) OR ((TI ~ ""природопозитивен"" OR TE ~ ..."
3,3,1,ALL_ALL,cs,DTS_SUBDOM = ALL_ALL,"TI, TE",7,"[návrh zahrnující přírodu, pozitivní pro přírodu, řešení založená na přírodě, strategie v oblasti biologické rozmanitosti, strategie EU v oblasti biologické rozmanitosti, obnova přírody, obnova ekosystémů]",návrh zahrnující přírodu | pozitivní pro přírodu | řešení založená na přírodě | strategie v oblasti biologické rozmanitosti | strategie EU v oblasti biologické rozmanitosti | obnova přírody | obnova ekosystémů,"DTS_SUBDOM = ALL_ALL AND (((TI ~ ""návrh zahrnující přírodu"" OR TE ~ ""návrh zahrnující přírodu"")) OR ((TI ~ ""pozitivní pro přírodu"" OR TE ~ ""pozitivní pro přírodu"")) OR ((TI ~ ""řešení založená na přírodě"" OR TE ~ ""řeš..."
4,4,1,ALL_ALL,da,DTS_SUBDOM = ALL_ALL,"TI, TE",8,"[naturinkluderende, naturinkluderende design, naturpositiv, naturbaserede løsninger, biodiversitetsstrategi, EU's biodiversitetsstrategi, genopretning af natur, genopretning af biodiversitet]",naturinkluderende | naturinkluderende design | naturpositiv | naturbaserede løsninger | biodiversitetsstrategi | EU's biodiversitetsstrategi | genopretning af natur | genopretning af biodiversitet,"DTS_SUBDOM = ALL_ALL AND (((TI ~ ""naturinkluderende"" OR TE ~ ""naturinkluderende"")) OR ((TI ~ ""naturinkluderende design"" OR TE ~ ""naturinkluderende design"")) OR ((TI ~ ""naturpositiv"" OR TE ~ ""naturpositiv"")) OR ((TI ~..."
5,5,1,ALL_ALL,de,DTS_SUBDOM = ALL_ALL,"TI, TE",10,"[naturverträgliche Planung, naturpositiv, naturbasierte Lösungen, naturbasierte Lösung, Netto-Gewinn, Netto-Gewinn für die Biodiversität, Biodiversitätsstrategie, EU-Biodiversitätsstrategie, Wiederherstellung der Nat...",naturverträgliche Planung | naturpositiv | naturbasierte Lösungen | naturbasierte Lösung | Netto-Gewinn | Netto-Gewinn für die Biodiversität | Biodiversitätsstrategie | EU-Biodiversitätsstrategie | Wiederherstellung ...,"DTS_SUBDOM = ALL_ALL AND (((TI ~ ""naturverträgliche Planung"" OR TE ~ ""naturverträgliche Planung"")) OR ((TI ~ ""naturpositiv"" OR TE ~ ""naturpositiv"")) OR ((TI ~ ""naturbasierte Lösungen"" OR TE ~ ""naturbasierte Lösungen""..."
6,6,1,ALL_ALL,el,DTS_SUBDOM = ALL_ALL,"TI, TE",9,"[φιλικός προς τη φύση σ

,job_index,chunk_index,scope,lang,term_count,fields,term_group
0,1,1,ALL_ALL,en,12,"TI, TE",nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
1,1,2,ALL_ALL,en,2,"TI, TE",nature repair | biodiversity strategy
2,2,1,ALL_ALL,bg,8,"TI, TE",интегриран с природата дизайн | природосъобразен дизайн | природопозитивен | природосъобразни решения | стратегия за биологичното разнообразие | стратегия на ЕС за биологичното разнообразие | възстановяване на природ...
3,3,1,ALL_ALL,cs,7,"TI, TE",návrh zahrnující přírodu | pozitivní pro přírodu | řešení založená na přírodě | strategie v oblasti biologické rozmanitosti | strategie EU v oblasti biologické rozmanitosti | obnova přírody | obnova ekosystémů
4,4,1,ALL_ALL,da,8,"TI, TE",naturinkluderende | naturinkluderende design | naturpositiv | naturbaserede løsninger | biodiversitetsstrategi | EU's biodiversitetsstrategi | genopretning af natur | genopretning af biodiversitet
5,5,1,ALL_ALL,de,10,"TI, TE",naturverträgliche Planung | naturpositiv | naturbasierte Lösungen | naturbasierte Lösung | Netto-Gewinn | Netto-Gewinn für die Biodiversität | Biodiversitätsstrategie | EU-Biodiversitätsstrategie | Wiederherstellung ...
6,6,1,ALL_ALL,el,9,"TI, TE",φιλικός προς τη φύση σχεδιασμός | θετικός για τη φύση | λύσεις βασισμένες στη φύση | καθαρό περιβαλλοντικό κέρδος | καθαρό κέρδος για τη βιοποικιλότητα | στρατηγική για τη βιοποικιλότητα | στρατηγική της ΕΕ για τη βι...
7,7,1,ALL_ALL,es,11,"TI, TE",diseño respetuoso con la naturaleza | positivo para la naturaleza | soluciones basadas en la naturaleza | ganancia neta | ganancia neta en biodiversidad | ganancia neta para la biodiversidad | estrategia de biodivers...
8,8,1,ALL_ALL,et,8,"TI, TE",loodusega arvestav disain | looduspositiivne | looduspõhised lahendused | looduspõhine lahendus | elurikkuse strateegia | ELi elurikkuse strateegia | looduse taastamine | ökosüsteemide taastamine
9,9,1,ALL_ALL,fi,8,"TI, TE",luontoystävällinen suunnittelu | luontopositiivinen | luontoon perustuvat ratkaisut | luontopohjaiset ratkaisut | biodiversiteettistrategia | EU:n biodiversiteettistrategia | luonnon ennallistaminen | ekosysteemien e...


## Parser debug probe: run one small live SOAP query and inspect the raw XML


In [ ]:
debug_probe_status = "not_run"
debug_probe_raw = ""
debug_probe_rows = []
debug_probe_diag_df = pd.DataFrame()
debug_probe_snippets = []

if RUN_LIVE and RUN_PARSER_DEBUG_PROBE and not query_chunks.empty:
    debug_row = query_chunks.iloc[0]
    debug_terms = list(debug_row["terms"][:DEBUG_TERM_LIMIT]) if isinstance(debug_row["terms"], list) else []
    debug_query = build_expert_query(debug_row["expert_scope"], debug_terms, fields=FIELDS)
    debug_payload = build_soap_envelope(
        debug_query,
        page=1,
        page_size=DEBUG_PAGE_SIZE,
        lang=debug_row["lang"],
    )
    debug_probe_raw, debug_status_code, _debug_body = post_eurlex_ws(
        debug_payload,
        timeout=TIMEOUT_S,
        retry_5xx=RETRY_5XX,
        min_interval_s=0,
        debug=DEBUG,
    )
    if debug_probe_raw:
        DEBUG_XML_PATH.write_text(debug_probe_raw, encoding="utf-8")
        debug_probe_status = f"live_ok_status_{debug_status_code}"
        totalhits, numhits, debug_probe_rows, missing_celex = parse_searchresults(debug_probe_raw, lang_for_url="EN")
        debug_probe_diag_df = build_result_diagnostics(debug_probe_raw, preferred_lang="en", limit=3)
        debug_probe_snippets = extract_result_snippets(debug_probe_raw, limit=2, max_chars=3500)
        print("Saved raw XML to", DEBUG_XML_PATH)
        print("totalhits =", totalhits)
        print("numhits =", numhits)
        print("hits_parsed =", len(debug_probe_rows))
        print("missing_celex =", missing_celex)
    else:
        debug_probe_status = f"live_failed_status_{debug_status_code}"

print("debug_probe_status =", debug_probe_status)


## Inspect parser diagnostics


In [ ]:
preview_df("debug_probe_parsed_rows", pd.DataFrame(debug_probe_rows))
preview_df("debug_probe_diag_df", debug_probe_diag_df)
for idx, snippet in enumerate(debug_probe_snippets, start=1):
    print(f"--- result snippet {idx} ---")
    print(snippet)
    print()


## Run the full EUR-Lex WebService retrieval


In [5]:
retrieval_mode = "live_webservice" if RUN_LIVE else "cached_rows"
live_error = ""
live_rows = []

if RUN_LIVE:
    try:
        live_rows = run_eurlex_jobs(
            jobs,
            fields=FIELDS,
            terms_per_query=TERMS_PER_QUERY,
            page_size=PAGE_SIZE,
            page_size_candidates=PAGE_SIZE_CANDIDATES,
            max_pages=MAX_PAGES,
            min_interval_s=MIN_INTERVAL_S,
            timeout=TIMEOUT_S,
            retry_5xx=RETRY_5XX,
            debug=DEBUG,
        )
    except Exception as exc:
        live_error = str(exc)
        retrieval_mode = "cached_rows" if ALLOW_FALLBACK else "live_failed"
        print("Live EUR-Lex WebService retrieval failed.")
        print(live_error)

print("retrieval_mode =", retrieval_mode)
print("live_rows =", len(live_rows))



=== JOB scope=ALL_ALL lang=en terms=14 groups=2 fields=('TI', 'TE') ===

[EURLEX] group 1/2 | terms=12
[EURLEX] trying page_size=100
[EURLEX] POST 6.56s status=200 bytes=1691815
[EURLEX] page=1 totalhits=1411 numhits=100 hits_parsed=53 missing_celex=47
[EURLEX] POST 5.34s status=200 bytes=1959362
[EURLEX] page=2 numhits=100 hits_parsed=63 missing_celex=37
[EURLEX] POST 6.34s status=200 bytes=1746134
[EURLEX] page=3 numhits=100 hits_parsed=82 missing_celex=18
[EURLEX] POST 5.92s status=200 bytes=1860048
[EURLEX] page=4 numhits=100 hits_parsed=65 missing_celex=35
[EURLEX] POST 5.51s status=200 bytes=1796904
[EURLEX] page=5 numhits=100 hits_parsed=62 missing_celex=38
[EURLEX] POST 7.38s status=200 bytes=1627366
[EURLEX] page=6 numhits=100 hits_parsed=66 missing_celex=34
[EURLEX] POST 5.04s status=200 bytes=1584225
[EURLEX] page=7 numhits=100 hits_parsed=67 missing_celex=33
[EURLEX] POST 6.60s status=200 bytes=1661908
[EURLEX] page=8 numhits=100 hits_parsed=65 missing_celex=35
[EURLEX] PO

## Inspect retrieved CELEX hits


In [6]:
if retrieval_mode == "live_webservice" and live_rows:
    all_eu_rows_df = pd.DataFrame(live_rows)
elif ALLOW_FALLBACK and CACHED_ROWS.exists():
    print("Using retrieval-output fallback:", CACHED_ROWS)
    all_eu_rows_df = load_cached_eu_rows(CACHED_ROWS)
elif ALLOW_FALLBACK:
    print("Using last-resort corpus reconstruction fallback from all_docs_df.csv")
    fallback_terms = SEARCH_TERMS + [term for terms in TRANSLATED_TERMS.values() for term in terms]
    all_eu_rows_df = reconstruct_eu_rows_from_corpus(CANONICAL_ALL_DOCS, fallback_terms)
else:
    raise RuntimeError("No retrieval rows available and fallback disabled.")

preview_df("all_eu_rows_df", all_eu_rows_df)
if not all_eu_rows_df.empty:
    display(all_eu_rows_df[["scope", "lang"]].value_counts().rename("rows").reset_index().head(20))
    display(all_eu_rows_df[["celex", "title", "term_group"]].head(10))


all_eu_rows_df: shape=(20122, 8)


,source,scope,lang,term_group,title,celex,date,url
0,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,Opinion of the European Committee of the Regions – Designing nature credits: A framework to promote biodiversity and ecosystem services,52025IR0140,2025-10-14,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025IR0140
1,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,"COMMUNICATION FROM THE COMMISSION TO THE EUROPEAN PARLIAMENT, THE COUNCIL, THE EUROPEAN ECONOMIC AND SOCIAL COMMITTEE AND THE COMMITTEE OF THE REGIONS Roadmap towards Nature Credits",52025DC0374,2025-07-07,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025DC0374
2,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,Commission NoticeTable of contents – Technical guidance on applying the do no significant harm principle under the Social Climate Fund Regulation,52025XC01596,2025-03-25,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025XC01596
3,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,P9_TA(2023)0277 – Nature restoration – Amendments adopted by the European Parliament on 12 July 2023 on the proposal for a regulation of the European Parliament and of the Council on nature restoration (COM(2022)0304...,52023AP0277,2023-07-12,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52023AP0277
4,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,Proposal for a REGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL on nature restoration,52022PC0304,2022-06-22,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52022PC0304


,scope,lang,rows
0,ALL_ALL,en,3878
1,ALL_ALL,de,1186
2,ALL_ALL,it,1169
3,ALL_ALL,es,1166
4,ALL_ALL,nl,1108
5,ALL_ALL,pt,1093
6,ALL_ALL,da,1059
7,ALL_ALL,fr,1013
8,ALL_ALL,mt,865
9,ALL_ALL,el,807


,celex,title,term_group
0,52025IR0140,Opinion of the European Committee of the Regions – Designing nature credits: A framework to promote biodiversity and ecosystem services,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
1,52025DC0374,"COMMUNICATION FROM THE COMMISSION TO THE EUROPEAN PARLIAMENT, THE COUNCIL, THE EUROPEAN ECONOMIC AND SOCIAL COMMITTEE AND THE COMMITTEE OF THE REGIONS Roadmap towards Nature Credits",nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
2,52025XC01596,Commission NoticeTable of contents – Technical guidance on applying the do no significant harm principle under the Social Climate Fund Regulation,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
3,52023AP0277,P9_TA(2023)0277 – Nature restoration – Amendments adopted by the European Parliament on 12 July 2023 on the proposal for a regulation of the European Parliament and of the Council on nature restoration (COM(2022)0304...,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
4,52022PC0304,Proposal for a REGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL on nature restoration,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
5,52024AP0089,P9_TA(2024)0089 – Nature restoration – European Parliament legislative resolution of 27 February 2024 on the proposal for a regulation of the European Parliament and of the Council on nature restoration (COM(2022)030...,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
6,52022SC0167,COMMISSION STAFF WORKING DOCUMENT IMPACT ASSESSMENT Accompanying the proposal for a Regulation of the European Parliament and of the Council on nature restoration,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
7,32024R1991,Regulation (EU) 2024/1991 of the European Parliament and of the Council of 24 June 2024 on nature restoration and amending Regulation (EU) 2022/869 (Text with EEA relevance),nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
8,52024IE1184,Opinion of the European Economic and Social Committee – A comprehensive strategy for biodiversity at COP16: bringing all sectors together for a common goal (own-initiative opinion),nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...
9,52025IR1930,Opinion of the European Committee of the Regions – Climate adaptation in cities and regions: building the European Climate Adaptation Plan,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiv

## Build `all_eu_rows`, `df_raw`, and `df_docs`


In [7]:
all_eu_rows = all_eu_rows_df.to_dict(orient="records")
print("raw EU rows:", len(all_eu_rows))

df_raw, df_docs = build_eu_doc_tables(all_eu_rows_df)
preview_df("df_raw", df_raw)
preview_df("df_docs", df_docs)
print("unique EU docs:", len(df_docs))
print("unique CELEX full IDs:", int(df_docs["celex_full"].nunique()))
print("unique base CELEX IDs:", int(df_docs["celex"].nunique()))


raw EU rows: 20122
df_raw: shape=(20122, 12)


,source,scope,lang,term_group,title,celex,date,url,celex_full,celex_version,doc_key,url_fix
0,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,Opinion of the European Committee of the Regions – Designing nature credits: A framework to promote biodiversity and ecosystem services,52025IR0140,2025-10-14,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025IR0140,52025IR0140,,EU:52025IR0140,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025IR0140
1,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,"COMMUNICATION FROM THE COMMISSION TO THE EUROPEAN PARLIAMENT, THE COUNCIL, THE EUROPEAN ECONOMIC AND SOCIAL COMMITTEE AND THE COMMITTEE OF THE REGIONS Roadmap towards Nature Credits",52025DC0374,2025-07-07,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025DC0374,52025DC0374,,EU:52025DC0374,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025DC0374
2,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,Commission NoticeTable of contents – Technical guidance on applying the do no significant harm principle under the Social Climate Fund Regulation,52025XC01596,2025-03-25,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025XC01596,52025XC01596,,EU:52025XC01596,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52025XC01596
3,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,P9_TA(2023)0277 – Nature restoration – Amendments adopted by the European Parliament on 12 July 2023 on the proposal for a regulation of the European Parliament and of the Council on nature restoration (COM(2022)0304...,52023AP0277,2023-07-12,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52023AP0277,52023AP0277,,EU:52023AP0277,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52023AP0277
4,EU,ALL_ALL,en,nature-positive | nature positive | nature inclusive | nature-inclusive | nature-inclusive | nature inclusive design | nature based solutions | nature-based solutions | biodiversity net gain | biodiversity net-gain |...,Proposal for a REGULATION OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL on nature restoration,52022PC0304,2022-06-22,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52022PC0304,52022PC0304,,EU:52022PC0304,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52022PC0304


df_docs: shape=(2790, 12)


,doc_key,source,celex,celex_full,celex_version,url,url_fix,title,date,scopes,query_langs,query_term_groups
0,EU:01995A0805(01)-20131213,EU,01995A0805(01),01995A0805(01)-20131213,20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",2013-12-13,"[""ALL_ALL""]","[""de"", ""et"", ""it""]","[""loodusega arvestav disain | looduspositiivne | looduspõhised lahendused | looduspõhine lahendus | elurikkuse strateegia | ELi elurikkuse strateegia | looduse taastamine | ökosüsteemide taastamine"", ""naturverträglic..."
1,EU:01996A0312(01)-20130925,EU,01996A0312(01),01996A0312(01)-20130925,20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,Convention sur la protection des Alpes (convention alpine),2013-09-25,"[""ALL_ALL""]","[""el"", ""fr""]","[""aménagement respectueux de la nature | positif pour la nature | solutions fondées sur la nature | solution fondée sur la nature | gain net | gain net pour la biodiversité | stratégie pour la biodiversité | stratégi..."
2,EU:01996L0061-20060224,EU,01996L0061,01996L0061-20060224,20060224,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,Директива 96/61/ЕО на Съвета от 24 септември 1996 година за комплексно предотвратяване и контрол на замърсяването,2006-02-24,"[""ALL_ALL""]","[""bg""]","[""интегриран с природата дизайн | природосъобразен дизайн | природопозитивен | природосъобразни решения | стратегия за биологичното разнообразие | стратегия на ЕС за биологичното разнообразие | възстановяване на прир..."
3,EU:02002D0011(01)-20051118,EU,02002D0011(01),02002D0011(01)-20051118,20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),2005-11-18,"[""ALL_ALL""]","[""es"", ""fr""]","[""aménagement respectueux de la nature | positif pour la nature | solutions fondées sur la nature | solution fondée sur la nature | gain net | gain net pour la biodiversité | stratégie pour la biodiversité | stratégi..."
4,EU:02002D0011(01)-20060313,EU,02002D0011(01),02002D0011(01)-20060313,20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),2006-03-13,"[""ALL_ALL""]","[""es"", ""fr""]","[""aménagement respectueux de la nature | positif pour la nature | solutions fondées sur la nature | solution fondée sur la nature | gain net | gain net pour la biodiversité | stratégie pour la biodiversité | stratégi..."


unique EU docs: 2790
unique CELEX full IDs: 2790
unique base CELEX IDs: 2628


## Deduplicate CELEX list for full-text retrieval


In [13]:
fulltext_index_df = df_docs[["doc_key", "celex_full", "celex", "celex_version", "title", "url", "url_fix", "query_langs"]].drop_duplicates(subset=["celex_full"]).copy()
fulltext_index_df = annotate_celex_types(fulltext_index_df, celex_col="celex_full")
preview_df("fulltext_index_df", fulltext_index_df)
print("CELEX documents queued before filtering:", len(fulltext_index_df))

celex_sector_counts_df = fulltext_index_df["celex_sector"].fillna("").astype(str).value_counts(dropna=False).rename_axis("celex_sector").reset_index(name="rows")
preview_df("celex_sector_counts_df", celex_sector_counts_df)

celex_descriptor_counts_df = fulltext_index_df["celex_descriptor"].fillna("").astype(str).value_counts(dropna=False).rename_axis("celex_descriptor").reset_index(name="rows")
preview_df("celex_descriptor_counts_df", celex_descriptor_counts_df)

celex_class_counts_df = fulltext_index_df["celex_class"].fillna("").astype(str).value_counts(dropna=False).rename_axis("celex_class").reset_index(name="rows")
preview_df("celex_class_counts_df", celex_class_counts_df)

nonstandard_celex_examples_df = fulltext_index_df[
    ~fulltext_index_df["celex_sector"].fillna("").astype(str).isin(["3"])
][[
    "celex_full", "celex_sector", "celex_sector_label", "celex_descriptor", "celex_descriptor_label", "celex_class", "title"
]].copy()
preview_df("nonstandard_celex_examples_df", nonstandard_celex_examples_df)

fulltext_index_filtered_df = filter_celex_types_for_fulltext(
    fulltext_index_df,
    mode=FULLTEXT_CELEX_FILTER_MODE,
    exclude_descriptors=FULLTEXT_EXCLUDE_DESCRIPTORS,
)
preview_df("fulltext_index_filtered_df", fulltext_index_filtered_df)
print("FULLTEXT_CELEX_FILTER_MODE =", FULLTEXT_CELEX_FILTER_MODE)
print("FULLTEXT_EXCLUDE_DESCRIPTORS =", FULLTEXT_EXCLUDE_DESCRIPTORS)
print("CELEX documents queued after filtering:", len(fulltext_index_filtered_df))

write_csv(fulltext_index_filtered_df, FULLTEXT_INDEX_OUT)


fulltext_index_df: shape=(2790, 18)


,doc_key,celex_full,celex,celex_version,title,url,url_fix,query_langs,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,EU:01995A0805(01)-20131213,01995A0805(01)-20131213,01995A0805(01),20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,"[""de"", ""et"", ""it""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
1,EU:01996A0312(01)-20130925,01996A0312(01)-20130925,01996A0312(01),20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,"[""el"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
2,EU:01996L0061-20060224,01996L0061-20060224,01996L0061,20060224,Директива 96/61/ЕО на Съвета от 24 септември 1996 година за комплексно предотвратяване и контрол на замърсяването,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,"[""bg""]",True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported
3,EU:02002D0011(01)-20051118,02002D0011(01)-20051118,02002D0011(01),20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
4,EU:02002D0011(01)-20060313,02002D0011(01)-20060313,02002D0011(01),20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported


CELEX documents queued before filtering: 2790
celex_sector_counts_df: shape=(8, 2)


,celex_sector,rows
0,5,1865
1,,394
2,3,235
3,0,207
4,9,41


celex_descriptor_counts_df: shape=(35, 2)


...

,celex_full,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_class,title
0,01995A0805(01)-20131213,,,,,unknown_or_invalid,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei..."
1,01996A0312(01)-20130925,,,,,unknown_or_invalid,Convention sur la protection des Alpes (convention alpine)
2,01996L0061-20060224,0,Consolidated texts,L,Directive,sector_0_consolidated,Директива 96/61/ЕО на Съвета от 24 септември 1996 година за комплексно предотвратяване и контрол на замърсяването
3,02002D0011(01)-20051118,,,,,unknown_or_invalid,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE)
4,02002D0011(01)-20060313,,,,,unknown_or_invalid,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE)


fulltext_index_filtered_df: shape=(2790, 18)


,doc_key,celex_full,celex,celex_version,title,url,url_fix,query_langs,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,EU:01995A0805(01)-20131213,01995A0805(01)-20131213,01995A0805(01),20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,"[""de"", ""et"", ""it""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
1,EU:01996A0312(01)-20130925,01996A0312(01)-20130925,01996A0312(01),20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,"[""el"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
2,EU:01996L0061-20060224,01996L0061-20060224,01996L0061,20060224,Директива 96/61/ЕО на Съвета от 24 септември 1996 година за комплексно предотвратяване и контрол на замърсяването,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,"[""bg""]",True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported
3,EU:02002D0011(01)-20051118,02002D0011(01)-20051118,02002D0011(01),20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
4,EU:02002D0011(01)-20060313,02002D0011(01)-20060313,02002D0011(01),20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported


FULLTEXT_CELEX_FILTER_MODE = all
FULLTEXT_EXCLUDE_DESCRIPTORS = []
CELEX documents queued after filtering: 2790
Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\eurlex_all_fulltext_index.csv
Shape: (2790, 18)


,doc_key,celex_full,celex,celex_version,title,url,url_fix,query_langs,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,EU:01995A0805(01)-20131213,01995A0805(01)-20131213,01995A0805(01),20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,"[""de"", ""et"", ""it""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
1,EU:01996A0312(01)-20130925,01996A0312(01)-20130925,01996A0312(01),20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,"[""el"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
2,EU:01996L0061-20060224,01996L0061-20060224,01996L0061,20060224,Директива 96/61/ЕО на Съвета от 24 септември 1996 година за комплексно предотвратяване и контрол на замърсяването,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,"[""bg""]",True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported
3,EU:02002D0011(01)-20051118,02002D0011(01)-20051118,02002D0011(01),20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
4,EU:02002D0011(01)-20060313,02002D0011(01)-20060313,02002D0011(01),20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported


# Full-text retrieval

## Batch full-text retrieval

This is the primary ordinary-EU full-text stage. It now follows the simpler old working logic from `NID_Retrieval_Pipeline_EURLEX_NIM.ipynb` for EU CELEX documents:

1. derive language candidates with EN first and then `query_langs`
2. derive CELEX variants: original CELEX, then base CELEX if consolidated
3. fetch from the Cellar CELEX URL using the old Accept / Accept-Language / Accept-Max-Cs-Size headers
4. parse returned HTML/XHTML into text
5. accept the first text above the minimal threshold and cache it

`legal-content` / `LexUriServ` are not the main ordinary-EU route here anymore; they remain part of the separate NIM pipeline.


In [14]:
fulltext_index_df = pd.read_csv(FULLTEXT_INDEX_OUT)
fulltext_index_df = annotate_celex_types(fulltext_index_df, celex_col="celex_full")
preview_df("fulltext_index_df (reloaded)", fulltext_index_df)

fulltext_cache_diag_df, fulltext_cache_compare_df, fulltext_cache_files_df = fulltext_cache_validation_diagnostics(
    FULLTEXT_CACHE_DIR,
    success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
)
display(fulltext_cache_diag_df)
preview_df("fulltext_cache_compare_df", fulltext_cache_compare_df)

fulltext_cache_state_df, fulltext_cache_df = summarize_fulltext_cache_state(
    FULLTEXT_CACHE_DIR,
    fulltext_index_df,
    success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
)
fulltext_cache_state_df = annotate_celex_types(fulltext_cache_state_df, celex_col="celex_full")
preview_df("fulltext_cache_state_df", fulltext_cache_state_df)
display(
    pd.DataFrame(
        [
            {"metric": "total_input_celex", "value": int(fulltext_index_df["celex_full"].astype(str).nunique())},
            {"metric": "already_successful", "value": int(fulltext_cache_state_df["cache_state"].eq("successful").sum())},
            {"metric": "previously_failed", "value": int(fulltext_cache_state_df["cache_state"].eq("failed").sum())},
            {"metric": "still_pending", "value": int(fulltext_cache_state_df["cache_state"].eq("pending").sum())},
        ]
    )
)

failed_cache_df = load_failed_fulltext_attempts(
    FULLTEXT_CACHE_DIR,
    success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
)
failed_cache_df = annotate_celex_types(failed_cache_df, celex_col="celex_full") if not failed_cache_df.empty else failed_cache_df
preview_df("failed_cache_df", failed_cache_df)

successful_celex = set(fulltext_cache_state_df.loc[fulltext_cache_state_df["cache_state"].eq("successful"), "celex_full"].astype(str))
failed_celex = set(fulltext_cache_state_df.loc[fulltext_cache_state_df["cache_state"].eq("failed"), "celex_full"].astype(str))
next_run_df = fulltext_index_df.copy()
if FULLTEXT_RESUME:
    next_run_df = next_run_df.loc[~next_run_df["celex_full"].astype(str).isin(successful_celex)].copy()
    if not FULLTEXT_RETRY_FAILURES:
        next_run_df = next_run_df.loc[~next_run_df["celex_full"].astype(str).isin(failed_celex)].copy()
preview_df("next_run_df", next_run_df)
print("Resume enabled:", FULLTEXT_RESUME)
print("Retry failures:", FULLTEXT_RETRY_FAILURES)
print("CELEX queued for next run:", len(next_run_df))


fulltext_index_df (reloaded): shape=(2790, 18)


,doc_key,celex_full,celex,celex_version,title,url,url_fix,query_langs,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,EU:01995A0805(01)-20131213,01995A0805(01)-20131213,01995A0805(01),20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,"[""de"", ""et"", ""it""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
1,EU:01996A0312(01)-20130925,01996A0312(01)-20130925,01996A0312(01),20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,"[""el"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
2,EU:01996L0061-20060224,01996L0061-20060224,01996L0061,20060224,Директива 96/61/ЕО на Съвета от 24 септември 1996 година за комплексно предотвратяване и контрол на замърсяването,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996L0061-20060224,"[""bg""]",True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported
3,EU:02002D0011(01)-20051118,02002D0011(01)-20051118,02002D0011(01),20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
4,EU:02002D0011(01)-20060313,02002D0011(01)-20060313,02002D0011(01),20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported


,metric,value
0,cache_csv_rows,149
1,cache_csv_status_200,0
2,cache_csv_text_ge_threshold,0
3,existing_txt_files,16
4,txt_files_ge_threshold,16
5,csv_vs_file_inconsistencies,16


fulltext_cache_compare_df: shape=(100, 15)


,celex_full,celex,celex_version,lang,retrieval_status,retrieval_error,text_len,source_url,timestamp,file_text_len,text_path,file_exists,csv_success,file_success,inconsistent_csv_vs_files
0,01995A0805(01)-20131213,01995A0805(01),20131213,it,202,route_exhausted_after_202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A01995A0805%2801%29%3AIT%3AHTML,2026-04-08T07:51:23.314432+00:00,0,NaN,NaN,False,False,False
1,01996A0312(01)-20130925,01996A0312(01),20130925,fr,202,route_exhausted_after_202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A01996A0312%2801%29%3AFR%3AHTML,2026-04-08T07:51:23.314432+00:00,0,NaN,NaN,False,False,False
2,01996L0061-20060224,01996L0061,20060224,bg,202,route_exhausted_after_202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A01996L0061%3ABG%3AHTML,2026-04-08T07:51:23.314432+00:00,59271,c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\eurlex_all_fulltext_cache\text_cache\01996L0061-20060224.txt,True,False,True,True
3,02002D0011(01)-20051118,02002D0011(01),20051118,fr,202,route_exhausted_after_202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A02002D0011%2801%29%3AFR%3AHTML,2026-04-08T07:51:23.314432+00:00,0,NaN,NaN,False,False,False
4,02002D0011(01)-20060313,02002D0011(01),20060313,fr,202,route_exhausted_after_202,0,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A02002D0011%2801%29%3AFR%3AHTML,2026-04-08T07:51:23.314432+00:00,0,NaN,NaN,False,False,False


...

failed_cache_df: shape=(149, 20)


,celex_full,celex,celex_version,lang,text,source_url,retrieval_status,retrieval_error,text_len,timestamp,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,01995A0805(01)-20131213,01995A0805(01),20131213,en,,,0,unsupported_celex_type_for_fulltext,0,2026-04-07T13:16:44.506784+00:00,False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
1,01996A0312(01)-20130925,01996A0312(01),20130925,en,,,0,unsupported_celex_type_for_fulltext,0,2026-04-07T13:16:44.506784+00:00,False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
2,01996L0061-20060224,01996L0061,20060224,en,,https://eur-lex.europa.eu/LexUriServ/LexUriServ.do?uri=CELEX%3A01996L0061%3ABG%3AHTML,202,HTTP 202,0,2026-04-07T13:16:44.506784+00:00,True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported
3,02002D0011(01)-20051118,02002D0011(01),20051118,en,,,0,unsupported_celex_type_for_fulltext,0,2026-04-07T13:16:44.506784+00:00,False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
4,02002D0011(01)-20060313,02002D0011(01),20060313,en,,,0,unsupported_celex_type_for_fulltext,0,2026-04-07T13:16:44.506784+00:00,False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported


next_run_df: shape=(2774, 18)


,doc_key,celex_full,celex,celex_version,title,url,url_fix,query_langs,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,EU:01995A0805(01)-20131213,01995A0805(01)-20131213,01995A0805(01),20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,"[""de"", ""et"", ""it""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
1,EU:01996A0312(01)-20130925,01996A0312(01)-20130925,01996A0312(01),20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,"[""el"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
3,EU:02002D0011(01)-20051118,02002D0011(01)-20051118,02002D0011(01),20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
4,EU:02002D0011(01)-20060313,02002D0011(01)-20060313,02002D0011(01),20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,"[""es"", ""fr""]",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported
6,EU:02002O0010-20030101,02002O0010-20030101,02002O0010,20030101,Orientación del Banco Central Europeo de 5 de diciembre de 2002 sobre el régimen jurídico de la contabilidad y la elaboración de informes financieros en el sistema europeo de Bancos Centrales (BCE/2002/10) (2003/131/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002O0010-20030101,https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002O0010-20030101,"[""es""]",True,0,Consolidated texts,O,Order / Oral question,,False,True,sector_0_consolidated,supported


Resume enabled: True
Retry failures: True
CELEX queued for next run: 2774


## Single-CELEX deep trace

Use this section to inspect the exact full-text decision path for a few CELEX values before running the batch job.


In [16]:
trace_summaries = []
trace_frames = []
trace_results = []
for trace_celex in FULLTEXT_TRACE_CELEX_LIST:
    trace_row = pd.Series({"celex_full": trace_celex, "celex": trace_celex, "lang": "en", "query_langs": "en"})
    trace_df, trace_result = trace_eurlex_fulltext_decision(
        trace_celex,
        cache_dir=FULLTEXT_CACHE_DIR,
        use_cache=False,
        timeout_s=FULLTEXT_TRACE_SINGLE_TIMEOUT_S,
        retries=max(1, min(FULLTEXT_RETRIES, 2)),
        min_interval_s=0.0,
    )
    trace_result = dict(trace_result)
    trace_result.pop("full_text_raw", None)
    trace_result.pop("full_text_clean", None)
    trace_results.append(trace_result)
    trace_summaries.append({
        "celex_full": trace_celex,
        "candidate_langs": ", ".join(lang_candidates_from_row(trace_row)),
        "celex_variants": ", ".join(celex_variants(trace_celex)),
        "cellar_url": cellar_celex_url(trace_celex),
        "final_url": trace_result.get("text_source_url", ""),
        "status": trace_result.get("retrieval_status", 0),
        "text_len": trace_result.get("text_len", 0),
        "retrieval_error": trace_result.get("retrieval_error", ""),
    })
    if not trace_df.empty:
        trace_frames.append(trace_df)

trace_summary_df = pd.DataFrame(trace_summaries)
trace_results_df = pd.DataFrame(trace_results)
trace_attempts_df = pd.concat(trace_frames, ignore_index=True) if trace_frames else pd.DataFrame()
preview_df("trace_summary_df", trace_summary_df)
preview_df("trace_results_df", trace_results_df)
preview_df("trace_attempts_df", trace_attempts_df)


[EURLEX TEXT] TRACE CELEX=01996L0061 variant=01996L0061 lang=EN route=cellar
[EURLEX TEXT] TRACE CELEX=02002D0272 variant=02002D0272 lang=EN route=cellar
[EURLEX TEXT] TRACE CELEX=02003L0087 variant=02003L0087 lang=EN route=cellar
trace_summary_df: shape=(3, 8)


,celex_full,candidate_langs,celex_variants,cellar_url,final_url,status,text_len,retrieval_error
0,01996L0061,en,01996L0061,http://publications.europa.eu/resource/celex/01996L0061,http://publications.europa.eu/resource/celex/01996L0061,404,0,HTTP 404
1,02002D0272,en,02002D0272,http://publications.europa.eu/resource/celex/02002D0272,http://publications.europa.eu/resource/celex/02002D0272,404,0,HTTP 404
2,02003L0087,en,02003L0087,http://publications.europa.eu/resource/celex/02003L0087,http://publications.europa.eu/resource/celex/02003L0087,404,0,HTTP 404


trace_results_df: shape=(3, 28)


,celex,celex_full,celex_version,title,url,text_source_url,text_len,retrieval_status,retrieval_error,lang,lang_source_fulltext,fetch_seconds,fetched_from_cache,text_path,celex_variant_used,route_used,content_type,attempt_trace,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support
0,01996L0061,01996L0061,,,,http://publications.europa.eu/resource/celex/01996L0061,0,404,HTTP 404,en,english_default,0.04,False,,01996L0061,cellar,,"[{'celex_full': '01996L0061', 'celex': '01996L0061', 'celex_variant': '01996L0061', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01996L0061', 'status': 404, 'error'...",True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported
1,02002D0272,02002D0272,,,,http://publications.europa.eu/resource/celex/02002D0272,0,404,HTTP 404,en,english_default,0.04,False,,02002D0272,cellar,,"[{'celex_full': '02002D0272', 'celex': '02002D0272', 'celex_variant': '02002D0272', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002D0272', 'status': 404, 'error'...",True,0,Consolidated texts,D,Decision,,False,True,sector_0_consolidated,supported
2,02003L0087,02003L0087,,,,http://publications.europa.eu/resource/celex/02003L0087,0,404,HTTP 404,en,english_default,0.04,False,,02003L0087,cellar,,"[{'celex_full': '02003L0087', 'celex': '02003L0087', 'celex_variant': '02003L0087', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02003L0087', 'status': 404, 'error'...",True,0,Consolidated texts,L,Directive,,False,True,sector_0_consolidated,supported


trace_attempts_df: shape=(3, 10)


,celex_full,celex,celex_variant,lang,route_name,final_url,status,error,content_type,text_len
0,01996L0061,01996L0061,01996L0061,en,cellar,http://publications.europa.eu/resource/celex/01996L0061,404,HTTP 404,,0
1,02002D0272,02002D0272,02002D0272,en,cellar,http://publications.europa.eu/resource/celex/02002D0272,404,HTTP 404,,0
2,02003L0087,02003L0087,02003L0087,en,cellar,http://publications.europa.eu/resource/celex/02003L0087,404,HTTP 404,,0


In [17]:
fulltext_live_df = pd.DataFrame()
fulltext_mode = "live_batch" if RUN_LIVE_FULLTEXT else "cached_fallback"
fulltext_live_error = ""

if RUN_LIVE_FULLTEXT:
    try:
        fulltext_live_df = batch_fetch_eurlex_fulltext(
            next_run_df,
            cache_dir=FULLTEXT_CACHE_DIR,
            use_cache=FULLTEXT_USE_CACHE,
            timeout_s=FULLTEXT_TIMEOUT_S,
            retries=FULLTEXT_RETRIES,
            min_interval_s=FULLTEXT_MIN_INTERVAL_S,
            max_docs=FULLTEXT_MAX_DOCS,
            verbose=FULLTEXT_VERBOSE,
            trace_routes=FULLTEXT_TRACE_ROUTES,
            resume=FULLTEXT_RESUME,
            retry_failures=FULLTEXT_RETRY_FAILURES,
            progress_every=FULLTEXT_PROGRESS_EVERY,
            cache_every=FULLTEXT_CACHE_EVERY,
            success_min_chars=FULLTEXT_SUCCESS_MIN_CHARS,
        )
    except Exception as exc:
        fulltext_live_error = str(exc)
        fulltext_mode = "cached_fallback" if ALLOW_FALLBACK else "live_failed"
        print("Live full-text retrieval failed.")
        print(fulltext_live_error)

print("fulltext_mode =", fulltext_mode)
preview_df("fulltext_live_df", fulltext_live_df)


=== EURLEX FULLTEXT RESUME STATE ===
Total input CELEX: 2774
Already successful: 0
Previously failed: 84
Still pending: 2690
Retry failures: True
Run set size: 2774
[EURLEX TEXT] 1/2774 CELEX=01995A0805(01) failed reason=404_not_found length=0
[EURLEX TEXT] 2/2774 CELEX=01996A0312(01) failed reason=404_not_found length=0
[EURLEX TEXT] 3/2774 CELEX=02002D0011(01) failed reason=404_not_found length=0
[EURLEX TEXT] 4/2774 CELEX=02002D0011(01) failed reason=404_not_found length=0
[EURLEX TEXT] 5/2774 CELEX=02002O0010 failed reason=sector0_route_mismatch length=0
[EURLEX TEXT] 5/2774 processed | success=0 | failed=5
[EURLEX TEXT] 6/2774 CELEX=02003R1725 success length=2156781
[EURLEX TEXT] 7/2774 CELEX=02003R1725 success length=2165833
[EURLEX TEXT] 8/2774 CELEX=02003R1725 success length=2179290
[EURLEX TEXT] 9/2774 CELEX=02003R1725 success length=2217797
[EURLEX TEXT] 10/2774 CELEX=02003R1725 success length=2223938
[EURLEX TEXT] 10/2774 processed | success=5 | failed=5
[EURLEX TEXT] 11/277

,celex,celex_full,celex_version,title,url,text_source_url,full_text_raw,full_text_clean,text_len,retrieval_status,retrieval_error,lang,lang_source_fulltext,fetch_seconds,fetched_from_cache,text_path,celex_variant_used,route_used,content_type,attempt_trace,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support,failure_category
0,01995A0805(01),01995A0805(01)-20131213,20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,http://publications.europa.eu/resource/celex/01995A0805(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,01995A0805(01),cellar,,"[{'celex_full': '01995A0805(01)-20131213', 'celex': '01995A0805(01)', 'celex_variant': '01995A0805(01)-20131213', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01995...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
1,01996A0312(01),01996A0312(01)-20130925,20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,http://publications.europa.eu/resource/celex/01996A0312(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,01996A0312(01),cellar,,"[{'celex_full': '01996A0312(01)-20130925', 'celex': '01996A0312(01)', 'celex_variant': '01996A0312(01)-20130925', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01996...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
2,02002D0011(01),02002D0011(01)-20051118,20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,http://publications.europa.eu/resource/celex/02002D0011(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002D0011(01),cellar,,"[{'celex_full': '02002D0011(01)-20051118', 'celex': '02002D0011(01)', 'celex_variant': '02002D0011(01)-20051118', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
3,02002D0011(01),02002D0011(01)-20060313,20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,http://publications.europa.eu/resource/celex/02002D0011(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002D0011(01),cellar,,"[{'celex_full': '02002D0011(01)-20060313', 'celex': '02002D0011(01)', 'celex_variant': '02002D0011(01)-20060313', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
4,02002O0010,02002O0010-20030101,20030101,Orientación del Banco Central Europeo de 5 de diciembre de 2002 sobre el régimen jurídico de la contabilidad y la elaboración de informes financieros en el sistema europeo de Bancos Centrales (BCE/2002/10) (2003/131/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002O0010-20030101,http://publications.europa.eu/resource/celex/02002O0010,,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002O0010,cellar,,"[{'celex_full': '02002O0010-20030101', 'celex': '02002O0010', 'celex_variant': '02002O0010-20030101', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002O0010-200301...",True,0,Consoli

## Build EU full-text dataframe


In [18]:
use_fallback = fulltext_live_df.empty or not fulltext_live_df["text_len"].gt(0).any()

if use_fallback and ALLOW_FALLBACK:
    print("Using cached full-text fallback from persisted EU full-text export.")
    eu_fulltext_df = load_eu_fulltext_docs(CANONICAL_ALL_DOCS)
elif use_fallback:
    raise RuntimeError("Live full-text retrieval produced no usable texts and fallback is disabled.")
else:
    eu_fulltext_df = fulltext_live_df.copy()
    if FALLBACK_FILL_MISSING_TEXT and ALLOW_FALLBACK and eu_fulltext_df["text_len"].fillna(0).eq(0).any():
        cached_fulltext_df = load_eu_fulltext_docs(CANONICAL_ALL_DOCS)[["celex_full", "full_text_clean", "text_len"]].drop_duplicates(subset=["celex_full"])
        eu_fulltext_df = eu_fulltext_df.merge(cached_fulltext_df, on="celex_full", how="left", suffixes=("", "_cached"))
        fill_mask = eu_fulltext_df["text_len"].fillna(0).eq(0) & eu_fulltext_df["text_len_cached"].fillna(0).gt(0)
        eu_fulltext_df.loc[fill_mask, "full_text_clean"] = eu_fulltext_df.loc[fill_mask, "full_text_clean_cached"]
        eu_fulltext_df.loc[fill_mask, "text_len"] = eu_fulltext_df.loc[fill_mask, "text_len_cached"]
        eu_fulltext_df.loc[fill_mask, "retrieval_error"] = eu_fulltext_df.loc[fill_mask, "retrieval_error"].fillna("") + " | filled_from_cached_export"
        eu_fulltext_df = eu_fulltext_df.drop(columns=[col for col in ["full_text_clean_cached", "text_len_cached"] if col in eu_fulltext_df.columns])

preview_df("eu_fulltext_df", eu_fulltext_df)
display(
    pd.DataFrame(
        [
            {"metric": "EU full-text rows", "value": len(eu_fulltext_df)},
            {"metric": "Rows with non-empty text", "value": int(eu_fulltext_df["text_len"].fillna(0).gt(0).sum())},
            {"metric": "Rows with empty text", "value": int(eu_fulltext_df["text_len"].fillna(0).eq(0).sum())},
        ]
    )
)


eu_fulltext_df: shape=(2774, 31)


,celex,celex_full,celex_version,title,url,text_source_url,full_text_raw,full_text_clean,text_len,retrieval_status,retrieval_error,lang,lang_source_fulltext,fetch_seconds,fetched_from_cache,text_path,celex_variant_used,route_used,content_type,attempt_trace,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support,failure_category
0,01995A0805(01),01995A0805(01)-20131213,20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,http://publications.europa.eu/resource/celex/01995A0805(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,01995A0805(01),cellar,,"[{'celex_full': '01995A0805(01)-20131213', 'celex': '01995A0805(01)', 'celex_variant': '01995A0805(01)-20131213', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01995...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
1,01996A0312(01),01996A0312(01)-20130925,20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,http://publications.europa.eu/resource/celex/01996A0312(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,01996A0312(01),cellar,,"[{'celex_full': '01996A0312(01)-20130925', 'celex': '01996A0312(01)', 'celex_variant': '01996A0312(01)-20130925', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01996...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
2,02002D0011(01),02002D0011(01)-20051118,20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,http://publications.europa.eu/resource/celex/02002D0011(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002D0011(01),cellar,,"[{'celex_full': '02002D0011(01)-20051118', 'celex': '02002D0011(01)', 'celex_variant': '02002D0011(01)-20051118', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
3,02002D0011(01),02002D0011(01)-20060313,20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,http://publications.europa.eu/resource/celex/02002D0011(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002D0011(01),cellar,,"[{'celex_full': '02002D0011(01)-20060313', 'celex': '02002D0011(01)', 'celex_variant': '02002D0011(01)-20060313', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
4,02002O0010,02002O0010-20030101,20030101,Orientación del Banco Central Europeo de 5 de diciembre de 2002 sobre el régimen jurídico de la contabilidad y la elaboración de informes financieros en el sistema europeo de Bancos Centrales (BCE/2002/10) (2003/131/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002O0010-20030101,http://publications.europa.eu/resource/celex/02002O0010,,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002O0010,cellar,,"[{'celex_full': '02002O0010-20030101', 'celex': '02002O0010', 'celex_variant': '02002O0010-20030101', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002O0010-200301...",True,0,Consoli

,metric,value
0,EU full-text rows,2774
1,Rows with non-empty text,1678
2,Rows with empty text,1096


## Save EU retrieval outputs


In [21]:
write_csv(eu_fulltext_df, OUT_FULLTEXT)

Wrote: c:\Users\acali\OneDrive - Danmarks Tekniske Universitet\PostDoc\Code\NiD-Policy-Analysis\analysis_pipeline\outputs\01A_eu_all_fulltext_docs_new.csv
Shape: (2774, 31)


,celex,celex_full,celex_version,title,url,text_source_url,full_text_raw,full_text_clean,text_len,retrieval_status,retrieval_error,lang,lang_source_fulltext,fetch_seconds,fetched_from_cache,text_path,celex_variant_used,route_used,content_type,attempt_trace,celex_valid,celex_sector,celex_sector_label,celex_descriptor,celex_descriptor_label,celex_notes,celex_is_corrigendum,celex_is_consolidated,celex_class,fulltext_support,failure_category
0,01995A0805(01),01995A0805(01)-20131213,20131213,"Convenzione sulla protezione e l’utilizzazione dei corsi d’acqua transfrontalieri e dei laghi internazionali in data, a Helsinki, il 17 marzo 1992 Nazioni unite 1992 Convenzione sulla protezione e l’utilizzazione dei...",https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01995A0805(01)-20131213,http://publications.europa.eu/resource/celex/01995A0805(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,01995A0805(01),cellar,,"[{'celex_full': '01995A0805(01)-20131213', 'celex': '01995A0805(01)', 'celex_variant': '01995A0805(01)-20131213', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01995...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
1,01996A0312(01),01996A0312(01)-20130925,20130925,Convention sur la protection des Alpes (convention alpine),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:01996A0312(01)-20130925,http://publications.europa.eu/resource/celex/01996A0312(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,01996A0312(01),cellar,,"[{'celex_full': '01996A0312(01)-20130925', 'celex': '01996A0312(01)', 'celex_variant': '01996A0312(01)-20130925', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/01996...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
2,02002D0011(01),02002D0011(01)-20051118,20051118,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20051118,http://publications.europa.eu/resource/celex/02002D0011(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002D0011(01),cellar,,"[{'celex_full': '02002D0011(01)-20051118', 'celex': '02002D0011(01)', 'celex_variant': '02002D0011(01)-20051118', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
3,02002D0011(01),02002D0011(01)-20060313,20060313,Décision de la Banque centrale européenne du 5 décembre 2002 concernant les comptes annuels de la Banque centrale européenne (BCE/2002/11) (2003/132/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002D0011(01)-20060313,http://publications.europa.eu/resource/celex/02002D0011(01),,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002D0011(01),cellar,,"[{'celex_full': '02002D0011(01)-20060313', 'celex': '02002D0011(01)', 'celex_variant': '02002D0011(01)-20060313', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002...",False,,,,,Pattern mismatch,False,False,unknown_or_invalid,supported,404_not_found
4,02002O0010,02002O0010-20030101,20030101,Orientación del Banco Central Europeo de 5 de diciembre de 2002 sobre el régimen jurídico de la contabilidad y la elaboración de informes financieros en el sistema europeo de Bancos Centrales (BCE/2002/10) (2003/131/CE),https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:02002O0010-20030101,http://publications.europa.eu/resource/celex/02002O0010,,,0,404,HTTP 404,en,english_fallback,0.04,False,,02002O0010,cellar,,"[{'celex_full': '02002O0010-20030101', 'celex': '02002O0010', 'celex_variant': '02002O0010-20030101', 'lang': 'en', 'route_name': 'cellar', 'final_url': 'http://publications.europa.eu/resource/celex/02002O0010-200301...",True,0,Consoli